# Read the spec sheet, then ask a judge

*[`enrich/01`](01_hypothetical_classification.ipynb) classified products from their names alone. Give both sides the product's features, then let a decision model choose among real categories.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omnifroodle/couchbase_notebooks/blob/main/notebooks/enrich/03_read_the_spec_sheet.ipynb)
[![Open in GitHub Codespaces](https://img.shields.io/badge/Open%20in%20Codespaces-2f363d?logo=github&logoColor=white)](https://codespaces.new/omnifroodle/couchbase_notebooks?quickstart=1)

**Claim.** A decision model choosing among real categories classifies better than a hallucinated path snapped to its nearest neighbour — and given a shortlist from a plain vector search, with no chat model in the pipeline at all, it still beats the hallucination.
**Result.** In the run stored here, on 300 WANDS products: the product's features lift hallucinate-then-resolve from 24.3% to 30.7% of exact paths, and help the no-LLM baseline less (20.0% → 18.0% exact, 53.7% → 59.3% department). Jev choosing from the ten real paths nearest the hallucinated one reaches 43.3% exact — 63 products fixed, 25 broken, p < 0.001. Choosing from the ten nearest the product itself, with no chat model, reaches 37.7% (59 fixed, 38 broken, p = 0.042). Department accuracy barely moves: the judge fixes the leaf, not the branch. It adds about three cents per thousand products and under two-tenths of a second.
**Requires.** couchbase · llm · local-embeddings · decision-model
**Read** ~12 min · **Run** ~19 min · **Cost** around a dollar — 900 chat-model calls; the 865 decision-model calls add about three cents

[`enrich/01`](01_hypothetical_classification.ipynb) classified products into WANDS's 1,623
categories without ever showing a model the list: invent a plausible category path, then snap it
to the nearest real one with a vector search. It beat embedding the product name directly by a
wide margin, and it left two things open.

**It read only the product's name.** Every WANDS product also carries a list of features —
materials, dimensions, intended use — and a five-word name is thin evidence for either method.
If the features help the baseline more than they help the hallucination, part of `enrich/01`'s
gap was the name being thin, not the technique working.

**Its last step is a nearest neighbour.** The resolution takes whichever real path sits closest
to the invented one in embedding space, which is how a sensible guess ends up one shelf over. A
second model that reads the product and *chooses* among a few real paths might do better than
distance. [`experiments/01`](../experiments/01_first_look_at_decision_models.ipynb) introduced
one built for that kind of question: Jev, a decision model that answers a `choice` with a
probability for every option, in a fraction of a second.

**What this notebook does**

1. Reruns `enrich/01`'s comparison on 300 products.
2. Gives both methods the product's features.
3. Measures how often the right category is even on a shortlist, built three ways.
4. Lets Jev pick from each shortlist, and compares the methods product by product.
5. Counts what each one costs.

In [ ]:
# Setup: load the helpers, install anything missing, check what this notebook needs.
import os
import pathlib
import subprocess
import sys

# Cloned on Colab, where there is no local checkout. Override to test a fork.
REPO_URL = os.environ.get("CBNB_REPO_URL", "https://github.com/omnifroodle/couchbase_notebooks")

try:
    import cbnb
except ModuleNotFoundError:
    here = pathlib.Path.cwd()
    root = next((p for p in [here, *here.parents] if (p / "cbnb" / "__init__.py").exists()), None)
    if root is None:
        # Colab: clone the repo so the committed datasets come with it. Once: a
        # restarted session keeps the files but not sys.path.
        root = pathlib.Path("cbnb-repo").resolve()
        if not root.exists():
            subprocess.check_call(["git", "clone", "--depth", "1", "--quiet", REPO_URL, str(root)])
    sys.path.insert(0, str(root))
    import cbnb

settings = cbnb.bootstrap(requires=["couchbase", "llm", "local-embeddings", "decision-model"])

## 1. What a product name leaves out

WANDS files every product under one path in its taxonomy and ships a list of features with each.
The name is all `enrich/01` classified from. Here is the rest, for one product in the sample.

In [2]:
# Load the taxonomy and 300 products, and show what one product's name leaves out.
from cbnb.datasets import Taxonomy, load_wands_products, load_wands_taxonomy

taxonomy = Taxonomy(load_wands_taxonomy())
products = load_wands_products()

N_EVAL = 300
sample = products.sample(N_EVAL, random_state=1729).reset_index(drop=True)
names = sample["product_name"].tolist()
features = [f.replace("|", "; ").strip("; ") for f in sample["product_features"]]
truth = sample["category_hierarchy"].tolist()

example = 2
print(f"name     : {names[example]}")
print(f"features : {features[example][:400]}")
print(f"filed as : {truth[example]}")

name     : callan newport single dummy door lever
features : whatsincluded : one door lever; doorlocation : interior; overallheight-toptobottom:2.63; overallwidth-sidetoside:4.12; countryoforigin : philippines; levershape : flat; function : single dummy; supplierintendedandapproveduse : non residential use; projection-fronttoback:2.37; warrantylength : lifetime; overallproductweight:0.75; rosetteshape : round; totalnumberoflevers:1; ul10ccompliant : no; ele
filed as : Home Improvement / Doors & Door Hardware / Door Hardware & Accessories / Door Levers / Flat Door Levers


## 2. The categories, in Couchbase

The same setup as `enrich/01`: every one of the 1,623 category paths becomes a document with an
embedding, behind one Search vector index. Every method below ends with a lookup against it, so
every answer is a category that exists.

In [3]:
# Embed every category path, store them in Couchbase, and index the vectors.
from cbnb.couchbase_io import connect, ensure_collection, ensure_vector_index, upsert_docs, wait_for_index
from cbnb.embeddings import Embedder

BUCKET, SCOPE, INDEX = settings.cb_bucket, "category_judge", "categories-vec"

embedder = Embedder()
cluster = connect(settings)
categories = ensure_collection(cluster, BUCKET, SCOPE, "categories")

category_vectors = embedder.encode(taxonomy.paths)
upsert_docs(categories, {
    f"cat::{i:04d}": {"path": path, "department": taxonomy.top_level(path),
                      "embedding": category_vectors[i].tolist()}
    for i, path in enumerate(taxonomy.paths)
}, progress=False)
ensure_vector_index(cluster, bucket_name=BUCKET, scope_name=SCOPE, collection_name="categories",
                    index_name=INDEX, vector_field="embedding", dims=embedder.dims,
                    keyword_fields=["path", "department"])
_ = wait_for_index(cluster, bucket_name=BUCKET, scope_name=SCOPE, index_name=INDEX,
                   expected=len(taxonomy.paths))

  indexed 1623/1623 (ready)   

  indexed 1623/1623 (ready)   

In [4]:
# Look up the nearest real category paths for many texts at once.
from concurrent.futures import ThreadPoolExecutor

from cbnb.couchbase_io import vector_search


def nearest(texts, k=1):
    """For each text, the k nearest real category paths, nearest first."""
    vectors = embedder.encode(texts)   # embed everything first; search in parallel after

    def one(vector):
        hits = vector_search(cluster, bucket_name=BUCKET, scope_name=SCOPE, index_name=INDEX,
                             vector_field="embedding", query_vector=vector, k=k,
                             fields=["path"], num_candidates=max(k * 4, 20))
        return [hit["path"] for hit in hits]

    with ThreadPoolExecutor(8) as pool:
        return list(pool.map(one, vectors))

In [5]:
# Score predictions three ways, and compare two methods on the products where they disagree.
from math import comb

import pandas as pd


def score(predictions, label):
    exact = sum(p == t for p, t in zip(predictions, truth)) / N_EVAL
    department = sum(taxonomy.top_level(p) == taxonomy.top_level(t)
                     for p, t in zip(predictions, truth)) / N_EVAL
    overlap = sum(taxonomy.prefix_overlap(p, t) for p, t in zip(predictions, truth)) / N_EVAL
    return pd.Series({"exact path": exact, "department": department, "prefix overlap": overlap},
                     name=label)


def versus(before, after, before_name, after_name):
    """Exact-path wins and losses on the products where two methods disagree."""
    right = lambda preds: [p == t for p, t in zip(preds, truth)]   # noqa: E731
    fixed = sum(not b and a for b, a in zip(right(before), right(after)))
    broke = sum(b and not a for b, a in zip(right(before), right(after)))
    n = fixed + broke
    # Exact two-sided sign test on the disagreements.
    p = min(1.0, 2 * sum(comb(n, i) for i in range(min(fixed, broke) + 1)) / 2 ** n) if n else 1.0
    return pd.Series({"fixed": fixed, "broke": broke, "p": p}, name=f"{after_name} vs {before_name}")


def table(rows):
    return pd.DataFrame(rows).style.format("{:.1%}")

## 3. `enrich/01`'s comparison, on 300 products

Embed the name and take the nearest category; or ask a chat model to invent a path from the name
and take the category nearest that. The prompt is `enrich/01`'s, word for word. Three scores, as
there: the **exact path**, the **department**, and the **prefix overlap** — how many levels of
the true path were matched from the top.

In [6]:
# Classify from the name alone, two ways: embed it, or hallucinate a path and resolve that.
from pydantic import BaseModel, Field

from cbnb.llm import LLM

CLASSIFY_SYSTEM = """\
You file products into the category taxonomy of a large home-goods retailer.

You do not have the taxonomy. Invent the category path you would expect it to contain.

Rules:
- 3 to 5 levels, most general first, joined by " / ".
- Level 1 is a department: Furniture, Outdoor, Lighting, Bed & Bath, Kitchen & Tabletop,
  Storage & Organization, Decor & Pillows, Appliances, Baby & Kids, Rugs, Pet, and so on.
- Each level narrows the one before it. The last level is the most specific.
- Use plural, title-cased retail category names ("Coffee Tables", not "a brown coffee table").
- Categories describe the kind of product, never its colour, material, brand or size.
- Output the path only. Do not explain."""


class Guess(BaseModel):
    category_path: str = Field(description="Full path from department to leaf, joined by ' / '")


llm_name = LLM()


def imagine(llm, text):
    return llm.structured(text, Guess, system=CLASSIFY_SYSTEM).category_path


name_text = [f"Product: {n}" for n in names]
embed_name = [hits[0] for hits in nearest(names)]
guess_name = llm_name.map(name_text, lambda t: imagine(llm_name, t), progress=False)
hallucinate_name = [hits[0] for hits in nearest(guess_name)]

table([score(embed_name, "embed the name"), score(hallucinate_name, "hallucinate from the name")])

,exact path,department,prefix overlap
embed the name,20.0%,53.7%,41.8%
hallucinate from the name,24.3%,72.0%,54.8%


The hallucination wins, as it did in `enrich/01`: 72.0% of departments right against 53.7%, and
24.3% of exact paths against 20.0%.

## 4. Give both sides the spec sheet

Same two methods, one change: each now sees the features. The embedding gets name and features as
one string — about 200 tokens, inside the 256 the local model reads. The chat model gets them as a
second line of its prompt.

In [7]:
# Classify again with the features: embed name and features, or hallucinate from both.
llm_features = LLM()

spec_text = [f"Product: {n}\nFeatures: {f}" for n, f in zip(names, features)]
embed_spec = [hits[0] for hits in nearest([f"{n} | {f}" for n, f in zip(names, features)])]
guess_spec = llm_features.map(spec_text, lambda t: imagine(llm_features, t), progress=False)
hallucinate_spec = [hits[0] for hits in nearest(guess_spec)]

table([score(embed_name, "embed the name"),
       score(embed_spec, "embed name + features"),
       score(hallucinate_name, "hallucinate from the name"),
       score(hallucinate_spec, "hallucinate from name + features")])

,exact path,department,prefix overlap
embed the name,20.0%,53.7%,41.8%
embed name + features,18.0%,59.3%,44.0%
hallucinate from the name,24.3%,72.0%,54.8%
hallucinate from name + features,30.7%,75.7%,61.1%


The features help the hallucination more than the embedding. Inventing from name and features
takes exact paths from 24.3% to 30.7%. The embedding gains departments (53.7% → 59.3%) and loses
exact paths (20.0% → 18.0%): a long string of specs pulls the vector towards the right department
and away from any one leaf.

So `enrich/01`'s result was not the name being thin. With more to read, the hallucination is a
little less far ahead on departments and further ahead on exact paths — 63 products right that
the embedding gets wrong, against 25 the other way, p < 0.001 on the products where they
disagree.

## 5. Was the right category in reach?

The nearest neighbour is one candidate. A judge needs several, with the right one among them, or
it has nothing to pick. Three ways to build a shortlist of real paths:

- **Three guesses.** Ask the chat model for three different paths, most likely first, and resolve
  each.
- **Nearest the product.** The ten paths nearest the name + features embedding. No chat model at
  all.
- **Nearest the guess.** The ten paths nearest the single hallucinated path from section 4.

In [8]:
# How often is the true category among the 10 nearest paths, or among three guesses?
K = 10


class ThreeGuesses(BaseModel):
    category_paths: list[str] = Field(
        description="Three different category paths, most likely first, each from department to leaf")


THREE_SYSTEM = CLASSIFY_SYSTEM.replace(
    "Invent the category path you would expect it to contain.",
    "Invent three different category paths it could plausibly be filed under: the most likely "
    "first, then two alternatives a retailer might choose instead.")

llm_three = LLM()
three = llm_three.map(spec_text, lambda t: llm_three.structured(t, ThreeGuesses, system=THREE_SYSTEM),
                      progress=False)
three_paths = [g.category_paths[:3] for g in three]

# Resolve every guess, then keep each product's distinct real paths in order.
flat = nearest([p for paths in three_paths for p in paths])
resolved_three, at = [], 0
for paths in three_paths:
    resolved_three.append(list(dict.fromkeys(hit[0] for hit in flat[at:at + len(paths)])))
    at += len(paths)

near_spec = nearest([f"{n} | {f}" for n, f in zip(names, features)], k=K)
near_guess = nearest(guess_spec, k=K)

reach = pd.Series({
    "the three guesses, resolved": sum(t in c for t, c in zip(truth, resolved_three)) / N_EVAL,
    f"the {K} paths nearest name + features": sum(t in c for t, c in zip(truth, near_spec)) / N_EVAL,
    f"the {K} paths nearest the hallucinated path": sum(t in c for t, c in zip(truth, near_guess)) / N_EVAL,
}, name="true category among the candidates")
print(f"distinct real paths from three guesses: {sum(map(len, resolved_three)) / N_EVAL:.1f} per product")
reach.to_frame().style.format("{:.1%}")

distinct real paths from three guesses: 2.4 per product


,true category among the candidates
"the three guesses, resolved",38.3%
the 10 paths nearest name + features,54.7%
the 10 paths nearest the hallucinated path,70.3%


Reach is the ceiling on anything a judge can do. The ten paths around the hallucinated one hold
the right category for 70.3% of products, against 30.7% for the nearest path alone: more than
half of the hallucination's misses land within ten neighbours of the answer. The three guesses
rarely do. They resolve to only 2.4 different real paths per product, and the right one is among
them 38.3% of the time. The ten paths around the product itself, with no chat model involved,
reach 54.7%.

## 6. Ask a judge

Jev reads the product's name and features and picks one path from the shortlist, with the same
`choice` question every time. With a single distinct path there is nothing to ask, so the
three-guess shortlist skips the call for products whose guesses all resolved to one place.

In [9]:
# Ask Jev to pick one real category from each shortlist, given the name and features.
import statistics

import httpx

from cbnb.decisions import choice, decide

QUESTION = "Which category would a large home-goods retailer file this product under?"


def judge(state, options, client):
    """Jev's pick from real category paths. One option needs no question."""
    if len(options) == 1:
        return options[0], None
    decision = decide(state, {"category": choice(QUESTION, {path: None for path in options})},
                      client=client)
    return decision.value("category"), decision


def judge_all(shortlists):
    with httpx.Client(timeout=20) as client, ThreadPoolExecutor(8) as pool:
        return list(pool.map(lambda args: judge(*args, client), zip(spec_text, shortlists)))


judged = {
    "Jev picks among three guesses": judge_all(resolved_three),
    f"Jev picks among the {K} nearest name + features": judge_all(near_spec),
    f"Jev picks among the {K} nearest the guess": judge_all(near_guess),
}
picks = {label: [pick for pick, _ in results] for label, results in judged.items()}

table([score(embed_spec, "embed name + features"),
       score(hallucinate_spec, "hallucinate from name + features"),
       *(score(predictions, label) for label, predictions in picks.items())])

,exact path,department,prefix overlap
embed name + features,18.0%,59.3%,44.0%
hallucinate from name + features,30.7%,75.7%,61.1%
Jev picks among three guesses,35.7%,78.3%,65.0%
Jev picks among the 10 nearest name + features,37.7%,76.0%,63.6%
Jev picks among the 10 nearest the guess,43.3%,78.7%,69.5%


In [10]:
# Compare each judged method with the best unjudged one, product by product.
best_label, best = "hallucinate from name + features", hallucinate_spec
pd.DataFrame([versus(embed_spec, best, "embed name + features", best_label),
              *(versus(best, predictions, best_label, label) for label, predictions in picks.items())]
             ).style.format({"fixed": "{:.0f}", "broke": "{:.0f}", "p": "{:.3f}"})

,fixed,broke,p
hallucinate from name + features vs embed name + features,63,25,0.000
Jev picks among three guesses vs hallucinate from name + features,29,14,0.032
Jev picks among the 10 nearest name + features vs hallucinate from name + features,59,38,0.042
Jev picks among the 10 nearest the guess vs hallucinate from name + features,63,25,0.000


The judge helps all three, by amounts that follow the reach.

**From the ten paths nearest the guess, exact paths go from 30.7% to 43.3%** — 63 products fixed,
25 broken, p < 0.001. Jev picks the right path for about three in five of the products where it
was on the list (43.3% of 70.3%).

**From the ten nearest the product, with no chat model anywhere, 37.7%.** That beats the
hallucination it replaces — 59 fixed, 38 broken, p = 0.042 — on a pipeline that is an embedding,
a vector search and one decision-model call.

**The three guesses gain least** because their shortlist was short of right answers. When the
right path was among them, Jev nearly always chose it (35.7% of 38.3%); it rarely was.

Departments barely move under any judge: 75.7% at best before, 78.7% at best after. The judge's
gain is in choosing the right leaf, not the right department.

## 7. What each one costs

In [11]:
# Chat-model calls and tokens, and Jev's calls, time and cost, for each method.
def chat_cost(llm):
    u = llm.usage
    return {"chat calls": u.calls + u.cached_calls, "chat tokens per product": (u.prompt_tokens + u.completion_tokens) / N_EVAL}


def jev_cost(results):
    asked = [d for _, d in results if d is not None]
    return {"Jev calls": len(asked),
            "Jev median ms": statistics.median(d.seconds for d in asked) * 1000 if asked else 0,
            "Jev $ per 1,000 products": sum(d.cost for d in asked) / N_EVAL * 1000}


none = {"chat calls": 0, "chat tokens per product": 0}
no_jev = {"Jev calls": 0, "Jev median ms": 0, "Jev $ per 1,000 products": 0}
labels = list(picks)
costs = pd.DataFrame({
    "embed the name": {**none, **no_jev},
    "embed name + features": {**none, **no_jev},
    "hallucinate from the name": {**chat_cost(llm_name), **no_jev},
    "hallucinate from name + features": {**chat_cost(llm_features), **no_jev},
    labels[0]: {**chat_cost(llm_three), **jev_cost(judged[labels[0]])},
    labels[1]: {**none, **jev_cost(judged[labels[1]])},
    labels[2]: {**chat_cost(llm_features), **jev_cost(judged[labels[2]])},
}).T
costs.style.format({"chat calls": "{:.0f}", "chat tokens per product": "{:,.0f}", "Jev calls": "{:.0f}",
                    "Jev median ms": "{:.0f}", "Jev $ per 1,000 products": "{:.3f}"})

,chat calls,chat tokens per product,Jev calls,Jev median ms,"Jev $ per 1,000 products"
embed the name,0,0,0,0,0.000
embed name + features,0,0,0,0,0.000
hallucinate from the name,300,289,0,0,0.000
hallucinate from name + features,300,444,0,0,0.000
Jev picks among three guesses,300,505,265,164,0.020
Jev picks among the 10 nearest name + features,0,0,300,158,0.031
Jev picks among the 10 nearest the guess,300,444,300,172,0.031


The judge is the cheap part. Jev costs about three cents per thousand products and answers in
under two-tenths of a second. The best method's real cost is the chat model behind the guess,
about 440 tokens a product. The no-chat pipeline — embed, retrieve ten, ask Jev — has no chat
model at all, and it is second-best here on exact paths.

## What to take from this

**Give the model the features.** Both methods can use them, and the hallucination gains more:
`enrich/01`'s lead held, and grew on exact paths, once the model had more to read.

**Nearest-neighbour is a weak last step.** More than half of the hallucination's misses had the
right path within its ten nearest neighbours. Taking the single nearest throws that away; a
judge that reads the product recovers some of it.

**A judge is only as good as its shortlist.** The order of the three judged methods is the order
of their reach. Spend effort on getting the right answer onto the list before spending it on the
judge.

**The cheapest pipeline here is respectable.** Embed the product, retrieve ten real categories,
ask Jev: no chat model, about three cents per thousand products, and better than the
hallucination it was set against.

**These are paired comparisons within one run.** A sign test on the products where two methods
disagree says a gap is unlikely to be chance among these products; it says nothing about how
far a model's answers move from one day to the next. The run card below says which models
answered here.

## Where to take this

- **Yes/no per candidate.** Jev can answer a yes/no question about each path instead of one
  `choice` across them. In [`experiments/02`](../experiments/02_is_this_the_same_customer.ipynb)
  the per-candidate form did better — for a task that needed a "none of these" answer.
  Classification always has an answer, so the single choice may suit it better; measure it.
- **Not: judge only when the resolution is unsure.** Asking Jev only when the nearest-neighbour
  score is low is the obvious saving. [`enrich/01`](01_hypothetical_classification.ipynb)'s
  bonus shows that score barely predicts which resolutions are right, and the judge costs less
  than the guess it corrects — so judge every product.
- **A better shortlist.** Ten paths around the guess reached 70.3%. Twenty, or the union of the
  guess's and the product's neighbours, may reach more — at the price of more distractors.
- **Use the leaf.** `enrich/01` section 7 shows what a category is for: a filter on a product
  search. An exact leaf is a much tighter filter than a department.

## What produced these results

Every number above came from the models below. A different model, or the same name served
differently, will give different numbers.

In [12]:
# Name the models that produced the results above.
cbnb.run_card()

> **The results above came from these models · 2026-10-02**
>
> | | model | how it ran |
> |---|---|---|
> | Chat model | `z-ai/glm-5.3` | via NanoGPT; 900 calls |
> | Embeddings | `sentence-transformers/all-MiniLM-L6-v2` | on this machine; 4,323 texts, 2,223 from cache |
> | Decision model | `typesafe/jev-1.13-20260917` | via OpenRouter; 865 calls |
>
> *NanoGPT and OpenRouter pass requests on to whichever host serves the model, so what answered may be a quantised or otherwise modified build of it.*
>
> *Cached results came from an earlier run of the same model.*

In [13]:
# Optional clean-up: delete what this notebook created. Uncomment to run.
# from cbnb.couchbase_io import drop_demo_data
# drop_demo_data(cluster, BUCKET, SCOPE)